1. for frontend map

creating lat and long data table


In [0]:
from pyspark.sql import Row

catalog = "databricks_7405612194732360"
gold_schema = "aqi_gold_layer"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}")

city_geo_data = [
    Row(city="Ahmedabad", lat=23.0225, lon=72.5714),
    Row(city="Aizawl", lat=23.7271, lon=92.7176),
    Row(city="Amaravati", lat=16.5745, lon=80.3736),
    Row(city="Amritsar", lat=31.6340, lon=74.8723),
    Row(city="Bengaluru", lat=12.9716, lon=77.5946),
    Row(city="Bhopal", lat=23.2599, lon=77.4126),
    Row(city="Brajrajnagar", lat=21.8204, lon=83.9180),
    Row(city="Chandigarh", lat=30.7333, lon=76.7794),
    Row(city="Chennai", lat=13.0827, lon=80.2707),
    Row(city="Coimbatore", lat=11.0168, lon=76.9558),
    Row(city="Delhi", lat=28.6139, lon=77.2090),
    Row(city="Ernakulam", lat=9.9816, lon=76.2999),
    Row(city="Gurugram", lat=28.4595, lon=77.0266),
    Row(city="Guwahati", lat=26.1445, lon=91.7362),
    Row(city="Hyderabad", lat=17.3850, lon=78.4867),
    Row(city="Jaipur", lat=26.9124, lon=75.7873),
    Row(city="Jorapokhar", lat=23.7070, lon=86.4147),
    Row(city="Kochi", lat=9.9312, lon=76.2673),
    Row(city="Kolkata", lat=22.5726, lon=88.3639),
    Row(city="Mumbai", lat=19.0760, lon=72.8777),
    Row(city="Patna", lat=25.5941, lon=85.1376),
    Row(city="Shillong", lat=25.5788, lon=91.8933),
    Row(city="Talcher", lat=20.9500, lon=85.2167),
    Row(city="Thiruvananthapuram", lat=8.5241, lon=76.9366),
    Row(city="Visakhapatnam", lat=17.6868, lon=83.2185)
]

df_city_geo = spark.createDataFrame(city_geo_data)

df_city_geo.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.gold_dim_city_geo")

print("City Geo Dimension table created successfully.")

BASE table

In [0]:
# AQI BUCKET FUNCTION

def add_aqi_bucket(df, col_name="aqi"):
    return df.withColumn(
        "aqi_bucket",
        when(col(col_name) <= 50, "Good")
        .when(col(col_name) <= 100, "Satisfactory")
        .when(col(col_name) <= 200, "Moderate")
        .when(col(col_name) <= 300, "Poor")
        .when(col(col_name) <= 400, "Very Poor")
        .otherwise("Severe")
    )

In [0]:
from pyspark.sql.functions import col, year, month, dayofmonth, avg, lower, trim
from pyspark.sql.functions import when

catalog = "databricks_7405612194732360"
silver_schema = "aqi_silver_layer_new"
gold_schema = "aqi_gold_layer"

city_day_table = f"{catalog}.{silver_schema}.city_day"
stations_table = f"{catalog}.{silver_schema}.stations"
geo_table = f"{catalog}.{gold_schema}.gold_dim_city_geo"

In [0]:
df_city = spark.read.table(city_day_table)

df_station = spark.read.table(stations_table) \
    .filter(col("Status") == "Active") \
    .select(lower(trim(col("City"))).alias("City"), "State")

df_geo = spark.read.table(geo_table) \
    .select(lower(trim(col("city"))).alias("City"), "lat", "lon")

# Join all
df_base = df_city \
    .withColumn("City", lower(trim(col("City")))) \
    .join(df_station, "City", "left") \
    .join(df_geo, "City", "left")

YEARLY CITY

In [0]:
spark.read.table(stations_table).printSchema()

In [0]:
df_yearly = df_base \
    .withColumn("year", year(col("Date"))) \
    .groupBy("City", "State", "lat", "lon", "year") \
    .agg(avg("AQI").alias("aqi"))

df_yearly = add_aqi_bucket(df_yearly, "aqi")

df_yearly.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.city_aqi_yearly")

print("✅ city_aqi_yearly created")

In [0]:
%sql
SELECT *
FROM databricks_7405612194732360.aqi_gold_layer.city_aqi_yearly 
ORDER BY city ASC, year ASC;

MONTHLY TABLE

In [0]:
df_monthly = df_base \
    .withColumn("year", year(col("Date"))) \
    .withColumn("month", month(col("Date"))) \
    .groupBy("City", "State", "lat", "lon", "year", "month") \
    .agg(avg("AQI").alias("aqi"))

df_monthly = add_aqi_bucket(df_monthly, "aqi")

df_monthly.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.city_aqi_monthly")

print("✅ city_aqi_monthly created")

In [0]:
%sql
SELECT *
FROM databricks_7405612194732360.aqi_gold_layer.city_aqi_monthly
ORDER BY city ASC, year ASC, month ASC;

DAILY TABLE

In [0]:
df_daily = df_base \
    .withColumn("year", year(col("Date"))) \
    .withColumn("month", month(col("Date"))) \
    .withColumn("day", dayofmonth(col("Date"))) \
    .groupBy("City", "State", "lat", "lon", "Date", "year", "month", "day") \
    .agg(avg("AQI").alias("aqi"))

df_daily = add_aqi_bucket(df_daily, "aqi")

df_daily.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.city_aqi_daily")

print("✅ city_aqi_daily created")

In [0]:
%sql
SELECT *
FROM databricks_7405612194732360.aqi_gold_layer.city_aqi_daily
ORDER BY city ASC, year ASC, month ASC, day ASC;

common


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

catalog = "databricks_7405612194732360"
silver_schema = "aqi_silver_layer_new"
gold_schema = "aqi_gold_layer"

city_day = f"{catalog}.{silver_schema}.city_day"
stations = f"{catalog}.{silver_schema}.stations"
geo = f"{catalog}.{gold_schema}.gold_dim_city_geo"

# Base DF (reuse everywhere)
df_city = spark.read.table(city_day)

df_station = spark.read.table(stations) \
    .filter(col("Status") == "Active") \
    .select(lower(trim(col("City"))).alias("City"), "State")

df_geo = spark.read.table(geo) \
    .select(lower(trim(col("city"))).alias("City"), "lat", "lon")

df_base = df_city \
    .withColumn("City", lower(trim(col("City")))) \
    .join(df_station, "City", "left") \
    .join(df_geo, "City", "left")

2.POWER BI DASHBOARD

2.1. TOP 10 POLLUTED CITIES

In [0]:
df_top10 = df_base \
    .withColumn("year", year(col("Date"))) \
    .groupBy("City", "State", "year") \
    .agg(round(avg("AQI"), 2).alias("avg_aqi"))

window_spec = Window.partitionBy("year").orderBy(col("avg_aqi").desc())

df_top10 = df_top10 \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 10)

df_top10 = add_aqi_bucket(df_top10, "avg_aqi")

df_top10.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.top_10_polluted_cities")

In [0]:
%sql
select * from databricks_7405612194732360.aqi_gold_layer.top_10_polluted_cities;

2.2. STATE POLLUTION SUMMARY

In [0]:
df_state = df_base \
    .withColumn("year", year(col("Date"))) \
    .groupBy("State", "year") \
    .agg(
        round(avg("AQI"), 2).alias("avg_aqi"),
        round(max("AQI"), 2).alias("max_aqi"),
        round(min("AQI"), 2).alias("min_aqi"),
        countDistinct("City").alias("num_cities")
    )

df_state = add_aqi_bucket(df_state, "avg_aqi")

df_state.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.state_pollution_summary")

In [0]:
%sql
select * from databricks_7405612194732360.aqi_gold_layer.state_pollution_summary;

2. 3. POLLUTION TREND (TIME SERIES)

In [0]:
df_trend = df_base \
    .withColumn("year", year(col("Date"))) \
    .withColumn("month", month(col("Date"))) \
    .groupBy("year", "month") \
    .agg(round(avg("AQI"), 2).alias("avg_aqi"))

df_trend = add_aqi_bucket(df_trend, "avg_aqi")

df_trend.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.pollution_trend_time_series")

In [0]:
%sql
select * from databricks_7405612194732360.aqi_gold_layer.pollution_trend_time_series;

2.4. POLLUTANT TREND BY CITY

In [0]:
df_pollutant = df_city \
    .withColumn("City", lower(trim(col("City")))) \
    .withColumn("year", year(col("Date"))) \
    .withColumn("month", month(col("Date"))) \
    .groupBy("City", "year", "month") \
    .agg(
        round(avg("`PM2.5`"), 2).alias("pm25"),
        round(avg("PM10"), 2).alias("pm10"),
        round(avg("NO2"), 2).alias("no2"),
        round(avg("CO"), 2).alias("co"),
        round(avg("O3"), 2).alias("o3")
    )

df_pollutant.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.pollutant_trend_by_city")

In [0]:
%sql
select * from databricks_7405612194732360.aqi_gold_layer.pollutant_trend_by_city;

2.5. AQI BUCKET DISTRIBUTION

In [0]:
df_bucket = df_base \
    .withColumn("year", year(col("Date"))) \
    .groupBy("year", "AQI_Bucket") \
    .agg(count("*").alias("count"))

window_spec = Window.partitionBy("year")

df_bucket = df_bucket \
    .withColumn("percentage", round(col("count") / sum("count").over(window_spec) * 100, 2))

df_bucket.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.aqi_bucket_distribution")

In [0]:
%sql
select * from databricks_7405612194732360.aqi_gold_layer.aqi_bucket_distribution;

2.6. CITY POLLUTION RANK OVER TIME

In [0]:
df_rank = df_base \
    .withColumn("year", year(col("Date"))) \
    .groupBy("City", "year") \
    .agg(round(avg("AQI"), 2).alias("avg_aqi"))

window_spec = Window.partitionBy("year").orderBy(col("avg_aqi").desc())

df_rank = df_rank \
    .withColumn("rank", dense_rank().over(window_spec))

df_rank = add_aqi_bucket(df_rank, "avg_aqi")

df_rank.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.city_pollution_rank_over_time")

In [0]:
%sql
select * from databricks_7405612194732360.aqi_gold_layer.city_pollution_rank_over_time;

helper for season data